In [1]:
import pandas as pd
import numpy as np
import pickle, os
import matplotlib.pyplot as plt
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
print("Libraries imported!")

Libraries imported!


In [2]:
df = pd.read_csv('cleaned_data.csv')
tokenized = df['clean_text'].apply(lambda x: str(x).split())
print(f"Articles : {len(tokenized):,}")
print(f"Sample : {tokenized[0][:10]}")

Articles : 44,223
Sample : ['ben', 'stein', 'call', 'th', 'circuit', 'court', 'committed', 'coup', 'dtat', 'constitution']


In [4]:
print("Training Word2Vec... please wait...")
w2v = Word2Vec(
    sentences=tokenized,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=10
)

print(f"Vocabulary size : {len(w2v.wv.key_to_index):,}")
print(f"Vector size : {w2v.vector_size}")
print(f"\nSample vector for 'trump':")
print(w2v.wv['trump'][:5])

Training Word2Vec... please wait...
Vocabulary size : 112,444
Vector size : 100

Sample vector for 'trump':
[ 3.3157747  -1.8276916   3.0812063   0.31087554 -1.930107  ]


In [6]:
# See which words are most similar to key words
print("Words most similar to 'trump':")
for word, score in w2v.wv.most_similar('trump', topn=5):
    print(f"{word:<20} {score:.4f}")

print("\nWords most similar to 'reuters':")
for word, score in w2v.wv.most_similar('reuters', topn=5):
    print(f"{word:<20} {score:.4f}")

Words most similar to 'trump':
presidentelect       0.6409
obama                0.5867
actually             0.5329
republican           0.5260
trumpthe             0.5187

Words most similar to 'reuters':
examinertrump        0.5588
postgallante         0.5563
dcthe                0.5383
postthe              0.4900
examinerht           0.4820


In [8]:
def get_avg_vector(text, model, vec_size=100):
    words = str(text).split()
    valid_words = [word for word in words if word in model.wv]
    if not valid_words:
        return np.zeros(vec_size)
    return np.mean(model.wv[valid_words], axis=0)

In [10]:
def get_avg_vector(text, w2v, vec_size=100):
    words = str(text).split()
    vectors = [w2v.wv[w] for w in words if w in w2v.wv]
    if len(vectors) == 0:
        return np.zeros(vec_size)
    return np.mean(vectors, axis=0)

print("Creating sentence vectors...")
X_w2v = np.array([
    get_avg_vector(t, w2v)
    for t in df['clean_text'].astype(str)
])

y = df['label'].values
print(f"Feature matrix shape: {X_w2v.shape}")
print(f"  {X_w2v.shape[0]:,} articles")
print(f"  {X_w2v.shape[1]} features per article")

Creating sentence vectors...
Feature matrix shape: (44223, 100)
  44,223 articles
  100 features per article


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
X_w2v, y,
test_size=0.2,
random_state=42,
stratify=y)
os.makedirs('saved_models', exist_ok=True)
w2v.save('saved_models/word2vec.model')
np.save('saved_models/X_train_w2v.npy', X_train)
np.save('saved_models/X_test_w2v.npy', X_test)
np.save('saved_models/y_train.npy', y_train)
np.save('saved_models/y_test.npy', y_test)
print("All files saved!")
print(f"Train: {X_train.shape}")
print(f"Test : {X_test.shape}")

All files saved!
Train: (35378, 100)
Test : (8845, 100)


In [12]:
print("=" * 50)
print(" FEATURE ENGINEERING SUMMARY — ESHAN")
print("=" * 50)
print(f"""
Method : Word2Vec
Vector Size : 100 dimensions
Window Size : 5
Min Count : 2
Epochs : 10
Vocab Size : {len(w2v.wv.key_to_index):,} words
Matrix Shape : {X_w2v.shape}
Train Size : {X_train.shape[0]:,}
Test Size : {X_test.shape[0]:,}
Files Saved:
saved_models/word2vec.model
saved_models/X_train_w2v.npy
saved_models/X_test_w2v.npy
saved_models/y_train.npy
saved_models/y_test.npy
""")

 FEATURE ENGINEERING SUMMARY — ESHAN

Method : Word2Vec
Vector Size : 100 dimensions
Window Size : 5
Min Count : 2
Epochs : 10
Vocab Size : 112,444 words
Matrix Shape : (44223, 100)
Train Size : 35,378
Test Size : 8,845
Files Saved:
saved_models/word2vec.model
saved_models/X_train_w2v.npy
saved_models/X_test_w2v.npy
saved_models/y_train.npy
saved_models/y_test.npy

